# Super-resolution — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def conv2d(x,k,pad=0,stride=1,dilation=1):
    x=np.asarray(x,float); k=np.asarray(k,float)
    if pad: x=np.pad(x,pad)
    kh,kw=k.shape; dh=(kh-1)*dilation+1; dw=(kw-1)*dilation+1; out=[]
    for i in range(0,x.shape[0]-dh+1,stride):
        row=[]
        for j in range(0,x.shape[1]-dw+1,stride): row.append(float(np.sum(x[i:i+dh:dilation,j:j+dw:dilation]*k)))
        out.append(row)
    return np.array(out)
def iou(a,b):
    ax1,ay1,ax2,ay2=a; bx1,by1,bx2,by2=b
    ix1,iy1=max(ax1,bx1),max(ay1,by1); ix2,iy2=min(ax2,bx2),min(ay2,by2)
    inter=max(0,ix2-ix1)*max(0,iy2-iy1); union=(ax2-ax1)*(ay2-ay1)+(bx2-bx1)*(by2-by1)-inter
    return inter/union if union else 0
def softmax(z):
    z=np.asarray(z,float); e=np.exp(z-z.max()); return e/e.sum()
def show(a,title,cmap='viridis'):
    plt.figure(figsize=(4,3)); plt.imshow(a,cmap=cmap); plt.colorbar(); plt.title(title)
print('setup ok')

## Super-resolution
Tiny CPU-only synthetic arrays for Super-resolution: no downloads, no pretrained models, and every cell ends with an assert.

In [ ]:
# 1) upsampling copies a tiny image into more pixels
low=np.array([[1,2],[3,4]]); up=np.repeat(np.repeat(low,2,0),2,1); show(up,'upsampled')
assert up.shape==(4,4) and up[1,1]==1

In [ ]:
# 2) flow vector field moves pixels right
flow=np.zeros((3,3,2)); flow[...,0]=1; y,x=np.mgrid[0:3,0:3]; plt.figure(figsize=(3,3)); plt.quiver(x,y,flow[...,0],flow[...,1]); plt.title('flow')
assert flow[1,1,0]==1

In [ ]:
# 3) keypoint heatmap peaks at a joint
x,y=np.meshgrid(np.arange(5),np.arange(5)); heat=np.exp(-((x-3)**2+(y-1)**2)/2); loc=np.unravel_index(np.argmax(heat),heat.shape); show(heat,'heatmap')
assert loc==(1,3)

In [ ]:
# 4) normalized embeddings compare by cosine
u=np.array([3.,4.]); u=u/np.linalg.norm(u); v=np.array([1.,0.]); sim=u@v; plt.figure(figsize=(4,3)); plt.bar(['cosine'],[sim]); plt.title('metric similarity')
assert abs(sim-.6)<1e-12

In [ ]:
# 5) video pools evidence over time
features=np.array([[1.,0.],[2.,1.],[3.,1.]]); pooled=features.mean(0); plt.figure(figsize=(4,3)); plt.bar(['f0','f1'],pooled); plt.title('temporal mean')
assert np.allclose(pooled,[2,2/3])